# Subset atlases for the dot plots

Keeps the 10 plotted genes, the cell-type column and the plotted cell types of each atlas, and writes small
`.h5ad` files to `results/scrna/` for `02_figures/Fig3_dotplots.ipynb`. For Tabula Sapiens only 10x 3' v3 cells
are used. Files are read from disk in blocks, so the 45 GB Tabula Sapiens file does not need to fit in memory.

In [ ]:
import sys; sys.path.insert(0, "../../utils")
from config import in_path, res_path

from contextlib import contextmanager

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from anndata.experimental import read_elem, sparse_dataset

In [ ]:
# Top 10 LASSO-selected protein-coding transcripts (Figure 3B), in plotting order
GENE_LIST = ["IGHD", "IGLC3", "MS4A1", "TDRD9", "SPDYC", "CLC", "ACCS", "FHDC1", "RGS16", "ALOX15"]

# Cells kept per atlas: obs filters, and the cell types plotted (in plotting order)
ATLASES = {
    "tabula_sapiens": dict(
        file="tabula_sapiens_all_cells_10df7690.h5ad",
        obs_filter={"assay": "10x 3' v3", "disease": "normal"},
        groupby="broad_cell_class",
        groups=[
            # immune / haematopoietic
            "lymphocyte of b lineage", "t cell", "granulocyte", "dendritic cell", "hematopoietic cell",
            # stromal / connective
            "fibroblast", "stromal cell", "connective tissue cell", "adventitial cell",
            # vascular
            "endothelial cell", "contractile cell",
            # epithelial
            "endo-epithelial cell", "meso-epithelial cell", "hepatocyte",
            # progenitor
            "stem cell",
        ],
    ),
    "li2025": dict(
        file="li2025_lbcl_atlas_777cf301.h5ad",
        obs_filter={},
        groupby="cell_type",
        groups=[
            "CD4-positive, alpha-beta T cell", "CD8-positive, alpha-beta T cell",
            "conventional dendritic cell", "natural killer cell", "macrophage", "monocyte",
            "fibroblast", "endothelial cell", "endothelial cell of lymphatic vessel",
        ],
    ),
    "strati": dict(
        file="strati2023_bmmc_4f515692.h5ad",
        obs_filter={},
        groupby="cell_type",
        groups=[
            "lymphocyte", "naive B cell", "memory B cell", "precursor B cell", "dendritic cell",
            "hematopoietic stem cell", "natural killer cell", "macrophage", "classical monocyte",
            "erythroid progenitor cell", "plasma cell", "common myeloid progenitor", "megakaryocyte",
        ],
    ),
}

In [ ]:
@contextmanager
def open_atlas(file):
    """obs, var and the on-disk (CSR) X of an h5ad file; layers, raw, embeddings and graphs are not read."""
    with h5py.File(in_path("scrna", file), "r") as f:
        yield read_elem(f["obs"]), read_elem(f["var"]), sparse_dataset(f["X"])


def read_gene_columns(X, cols, chunk=200_000):
    """X[:, cols] of an on-disk CSR matrix, read in blocks of `chunk` cells to limit memory."""
    return sp.vstack([X[i:i + chunk][:, cols] for i in range(0, X.shape[0], chunk)], format="csr")


def selected_cells(obs, obs_filter, groupby, groups):
    keep = obs[groupby].isin(groups)
    for col, value in obs_filter.items():
        keep &= obs[col] == value
    return keep.to_numpy()


def subset_atlas(file, obs_filter, groupby, groups, genes=GENE_LIST, gene_col="feature_name"):
    """Cells passing `obs_filter` in `groups` x `genes`, keeping only .X, var[gene_col] and obs[groupby]."""
    with open_atlas(file) as (obs, var, X):
        rows = selected_cells(obs, obs_filter, groupby, groups)
        cols = var[gene_col].isin(genes).to_numpy()
        assert set(var.loc[cols, gene_col]) == set(genes), "some genes not found"
        X_genes = read_gene_columns(X, np.flatnonzero(cols))

    sub = ad.AnnData(
        X=X_genes[np.flatnonzero(rows)],
        obs=pd.DataFrame(
            {groupby: pd.Categorical(obs.loc[rows, groupby].astype(str),
                                     categories=groups, ordered=True)},
            index=obs.index[rows]),
        var=pd.DataFrame({gene_col: var.loc[cols, gene_col].astype(str)},
                         index=var.index[cols]),
    )
    print(f"{file}: kept {rows.sum():,} of {len(obs):,} cells, {cols.sum()} genes")
    return sub


subsets = {name: subset_atlas(**spec) for name, spec in ATLASES.items()}

In [ ]:
for name, sub in subsets.items():
    sub.write_h5ad(res_path("scrna", f"{name}_dotplot.h5ad"), compression="gzip")
    print(name, sub.shape, sub.obs.iloc[:, 0].value_counts().min(), "cells in smallest group")